# Legacy exploratory notebook

Legacy exploratory notebook; not the approved OPEN-07 v5 object. Do not cite it for timing identification, QT amplification, final-holder distribution, a SOMA substitution mechanism, or structural causality. All numerical displays below are superseded exploratory material. The README and live site state the retained v5 headline and boundaries.

In [ ]:
# Environment setup
# Requires: ~/venvs/bidbridge (Python 3.11)
# Install kernel if needed: pip install jupyter ipykernel
# Then: python -m ipykernel install --user --name bidbridge --display-name "BidBridge"

%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})

In [ ]:
# Project paths
ROOT = Path.cwd().parent
DATA = ROOT / 'data' / 'processed'
FIGS = ROOT / 'outputs' / 'figures'
TBLS = ROOT / 'outputs' / 'tables'

## 1. Overview

This exploration describes aggregate auction supply and dealer-position associations. It does not identify excess supply, final-holder demand, dealer-capacity strain, or structural mechanisms.

In [ ]:
panel = pd.read_csv(DATA / 'auction_week_panel.csv', parse_dates=['week_start'])
print(f'Panel shape: {panel.shape[0]} weeks x {panel.shape[1]} columns')
print(f'Columns: {list(panel.columns)}')

---
## 2. Data Coverage

The panel merges five data sources at the weekly level. Not every source is available for every week, so coverage varies.

In [ ]:
date_min = panel['week_start'].min().strftime('%Y-%m-%d')
date_max = panel['week_start'].max().strftime('%Y-%m-%d')
print(f'Date range: {date_min}  to  {date_max}')
print(f'Total weeks: {len(panel)}')

In [ ]:
coverage = pd.DataFrame({
    'Source': ['Auctions (FiscalData)', 'PD positions (NY Fed)',
              'SOMA holdings (NY Fed)', 'H.8 bank holdings (FRED)',
              'Investor class (Treasury)'],
    'Key column': ['awarded_amount_total', 'pd_treasury_inventory',
                   'soma_treasury_total', 'bank_treasury_securities',
                   'dealer_share_allotment'],
})
coverage['Non-null weeks'] = coverage['Key column'].apply(
    lambda c: int(panel[c].notna().sum()))
coverage['Coverage %'] = (coverage['Non-null weeks'] / len(panel) * 100).round(1)
coverage

---
## 3. Supply Trends

Weekly auction supply has grown dramatically over the sample period, from roughly \$6 trillion per year to over \$30 trillion per year.

In [ ]:
fig, ax = plt.subplots()
ax.plot(panel['week_start'], panel['awarded_amount_total'] / 1e9,
        linewidth=0.7, color='steelblue')
ax.set_ylabel('Awarded amount ($B)')
ax.set_xlabel('Week')
ax.set_title('Weekly Treasury Auction Supply')
ax.axhline(panel['awarded_amount_total'].median() / 1e9,
           ls='--', color='grey', lw=0.8, label='median')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
annual = panel.set_index('week_start').resample('YE')['awarded_amount_total'].sum() / 1e12
print('Annual auction supply ($T):')
print(annual.to_string(float_format='{:.1f}'.format))

## 4. Dealer allotments

Supply and dealer allotment share are inversely correlated in this exploratory sample. This does not identify who absorbed marginal supply or who ultimately held the securities.

In [ ]:
display(Image(filename=str(FIGS / 'dealer_share_vs_supply.png'), width=700))

In [ ]:
mask = panel['dealer_share_allotment'].notna() & panel['awarded_amount_total'].notna()
r = panel.loc[mask, 'dealer_share_allotment'].corr(
    panel.loc[mask, 'awarded_amount_total'])
print(f'Pearson correlation (supply vs dealer share): r = {r:.2f}')

---
## 5. Bridge Episodes

A **bridge episode** is defined as a week where the dealer inventory change z-score exceeds +1: dealers absorbed significantly more inventory than their historical norm.

The panel flags 66 such episodes across the sample (roughly 7.8% of weeks). The table below summarizes episodes by year.

In [ ]:
n_episodes = int(panel['bridge_episode'].sum())
print(f'Total bridge episodes: {n_episodes}')
print(f'Share of sample: {n_episodes / len(panel) * 100:.1f}%')

In [ ]:
# Timeline of bridge episodes
fig, ax = plt.subplots(figsize=(14, 3))
ep = panel[panel['bridge_episode']]
ax.vlines(ep['week_start'], 0, 1, color='crimson', alpha=0.6, linewidth=1.5)
ax.set_yticks([])
ax.set_xlabel('Week')
ax.set_title(f'Bridge Episode Timeline ({n_episodes} episodes)')
plt.tight_layout()
plt.show()

In [ ]:
episode_summary = pd.read_csv(TBLS / 'bridge_episode_summary.csv')
episode_summary

Episodes have become larger over time. Average inventory change during episodes grew from ~\$18B in 2014 to ~\$53B in 2026, mirroring the growth in auction supply.

---
## 6. Event Studies

The pre-generated event study figures show dealer inventory dynamics around **refunding weeks** and **bridge episodes**.

In [ ]:
print('Refunding Week Event Study:')
display(Image(filename=str(FIGS / 'event_study_refunding.png'), width=700))

In [ ]:
print('Bridge Episode Event Study:')
display(Image(filename=str(FIGS / 'event_study_bridge_episodes.png'), width=700))

Refunding weeks see an average inventory buildup of ~\$10.4B compared to ordinary weeks (-\$1.8B). The Welch t-test confirms this is highly significant (t = 6.32, p < 0.001).

## 7. Maturity structure

Dealer allotment shares differ across maturity categories. These descriptive differences do not identify inventory costs, financing costs, or dealer preferences.

In [ ]:
mat = pd.read_csv(DATA / 'maturity_bucket_panel.csv', parse_dates=['week_start'])
print(f'Maturity panel: {mat.shape[0]} rows x {mat.shape[1]} columns')
print(f'Buckets: {sorted(mat["maturity_bucket"].unique())}')

In [ ]:
bucket_summary = pd.read_csv(TBLS / 'maturity_bucket_summary.csv')
cols = ['maturity_bucket', 'dealer_share_mean', 'dealer_share_std',
        'weighted_bid_to_cover_mean', 'bucket_share_of_weekly_mean']
bucket_summary[cols].sort_values('dealer_share_mean', ascending=False)

In [ ]:
display(Image(filename=str(FIGS / 'dealer_share_by_instrument.png'), width=700))

The maturity gradient is descriptive; it does not establish an inventory-cost mechanism or willingness to warehouse.

## 8. Stress regimes

The implemented QT flag uses announcement dates (October 2017–September 2019 and June 2022–December 2025). It is not defined by four consecutive declining SOMA weeks. The remaining flags are descriptive classifications, not identification of funding routes or capacity constraints.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from bidbridge.features.stress_flags import add_stress_flags

panel_s = add_stress_flags(panel)
flags = ['qt_period', 'tga_rebuild', 'weak_bank_absorption', 'risk_off_window']
print('Stress flag activation counts:')
print(panel_s[flags].sum().to_string())

In [ ]:
# Bridge rate during QT vs non-QT
qt_bridge = panel_s.loc[panel_s['qt_period'], 'bridge_episode'].mean()
non_qt_bridge = panel_s.loc[~panel_s['qt_period'], 'bridge_episode'].mean()
print(f'Bridge rate during QT:     {qt_bridge:.1%}')
print(f'Bridge rate outside QT:    {non_qt_bridge:.1%}')
print(f'Ratio:                     {qt_bridge / non_qt_bridge:.1f}x')

In [ ]:
stress_summary = pd.read_csv(TBLS / 'stress_regime_summary.csv')
stress_summary

In [ ]:
display(Image(filename=str(FIGS / 'stress_bridge_conditioning.png'), width=700))

Exploratory regime comparisons are not evidence of QT amplification, SOMA substitution, or an established regime difference.

---
## 9. Regression Results

Two OLS regressions model weekly dealer inventory change ($M):
- **Basic**: supply, dealer share, refunding dummy
- **Extended**: adds SOMA change, bank holdings change, and a linear trend (HC1 robust SEs)

In [ ]:
reg_basic = pd.read_csv(TBLS / 'regression_basic.csv')
print('=== Basic Regression ===')
reg_basic

In [ ]:
reg_ext = pd.read_csv(TBLS / 'regression_extended.csv')
print('=== Extended Regression ===')
reg_ext

### Exploratory regression output

The coefficients below are conditional associations in a superseded exploratory sample. They do not establish Fed/dealer substitution, structural trends in final-holder absorption, or causal inventory effects.

In [ ]:
ref_test = pd.read_csv(TBLS / 'regression_refunding_test.csv')
print('=== Refunding Week Welch t-test ===')
ref_test

## 10. Interpretation

The approved v5 evidence is the maturity-bucket and week fixed-effects association reported in the README. This notebook does not reproduce that object. Local projections fail clean timing validation; final holders, later distribution, and structural policy mechanisms remain unidentified.